# 2주차 · 2교시 통계적 추론 + 회귀 과제 — 모범답안

**과제 유형: A + B**

이 노트북은 과제 노트북과 동일한 구조이며, 모든 TODO/빈칸을 채운 실행 가능한 정답과 각 단계의 결과 해석을 담고 있습니다. (아래 seed는 예시로 `26`을 사용합니다. 실제 제출본은 각자의 `MY_SEED` 값에 따라 부트스트랩·train/test 관련 수치가 조금씩 달라집니다. 반면 전체 데이터로 적합하는 회귀·t검정 수치는 seed와 무관하게 고정입니다.)

## 0. 환경 설정

In [ ]:
# 기본 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# (선택) Colab에서 그래프의 한글 라벨이 네모(□)로 깨지지 않도록 한글 폰트 설정
!pip -q install koreanize-matplotlib
import koreanize_matplotlib
plt.rcParams["axes.unicode_minus"] = False

# 통계/모델링 라이브러리
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 예시 seed (제출 시 본인 값으로 교체)
MY_SEED = 26

# 참고: np.random.seed는 전역 기본 난수 재현용이며, 부트스트랩·CLT는
# np.random.default_rng(MY_SEED) 개별 생성기로, train/test는 random_state=MY_SEED로 재현한다.
np.random.seed(MY_SEED)
print('내 seed:', MY_SEED)

## 1. 필수 과제

### 1.1 데이터 로드

In [ ]:
tips = sns.load_dataset("tips")

print(tips.shape)
tips.head()

In [ ]:
# 결측치와 자료형 확인
tips.info()
print("\n결측치 개수:")
print(tips.isnull().sum())

# 수치형 기술통계
tips.describe()

**해석:** `tips`는 244행, 결측치가 없습니다. `total_bill`, `tip`, `size`는 수치형이고 `sex`, `smoker`, `day`, `time`은 범주형(category)입니다. 팁 평균은 약 3.0달러 수준입니다.

### 1.2 EDA / 상관

In [ ]:
num_cols = ["total_bill", "tip", "size"]
corr = tips[num_cols].corr()
print(corr)

plt.figure(figsize=(5,4))
plt.scatter(tips["total_bill"], tips["tip"], alpha=0.5)
plt.xlabel("total_bill")
plt.ylabel("tip")
plt.title("total_bill vs tip")
plt.show()

**[해석 질문 1] 답:** `total_bill`과 `tip`의 상관계수는 약 0.68로 뚜렷한 양의 상관입니다. 산점도에서도 청구액이 클수록 팁이 대체로 커지는 우상향 추세가 보여, 상관계수 값과 그림의 모양이 일치합니다. 다만 청구액이 커질수록 점들이 위아래로 더 넓게 퍼져(팁의 변동이 커져) 완벽한 직선은 아닙니다.

### 1.3 표본분포 · CLT 시뮬레이션 (유형 A)

In [ ]:
population = tips["total_bill"].values
pop_mean = population.mean()
print("모집단(가정) 평균:", round(pop_mean, 3))

rng = np.random.default_rng(MY_SEED)

def sample_means(pop, n, n_iter=2000):
    """pop에서 크기 n 표본을 n_iter번 뽑아 각 표본평균을 반환"""
    means = []
    for _ in range(n_iter):
        sample = rng.choice(pop, size=n, replace=True)
        means.append(sample.mean())
    return np.array(means)

# 마지막 열: 이론적 표준오차 σ/√n — 경험 표준편차와 거의 일치한다
for n in [2, 10, 30, 100]:
    m = sample_means(population, n=n)
    se_theory = population.std() / np.sqrt(n)
    print(f"n={n:3d} | 표본평균들의 평균={m.mean():.3f}, 경험 표준편차={m.std():.3f}, 이론 σ/√n={se_theory:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))

m_small = sample_means(population, n=2)
m_large = sample_means(population, n=30)

axes[0].hist(m_small, bins=30)
axes[0].set_title("표본평균 분포 (n=2)")
axes[1].hist(m_large, bins=30)
axes[1].set_title("표본평균 분포 (n=30)")
plt.show()

**[해석 질문 2] 답:** (a) 표본평균들의 표준편차는 `n=2`에서 크게 흔들리다가 `n`이 커질수록 눈에 띄게 작아졌습니다(대략 1/√n 비율로 감소). (b) `n=2`일 때는 원래 `total_bill`이 오른쪽으로 치우친 모양을 반영해 표본평균 분포도 비대칭이었지만, `n=30`에서는 좌우 대칭의 종 모양(정규분포)에 가까워졌습니다. 두 변화 모두 중심극한정리의 내용 — 표본이 커질수록 표본평균의 분포는 정규에 가까워지고 퍼짐은 좁아진다 — 을 그대로 보여 줍니다. 표본평균들의 평균은 `n`과 무관하게 모집단 평균(≈19.79) 근처에 머뭅니다.

### 1.4 신뢰구간 (유형 A · 부트스트랩)

In [ ]:
tip_values = tips["tip"].values
n = len(tip_values)
boot_means = []

boot_rng = np.random.default_rng(MY_SEED)

B = 5000
for _ in range(B):
    resample = boot_rng.choice(tip_values, size=n, replace=True)
    boot_means.append(resample.mean())

boot_means = np.array(boot_means)

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"팁 평균의 점추정치: {tip_values.mean():.3f}")
print(f"95% 부트스트랩 신뢰구간: [{ci_low:.3f}, {ci_high:.3f}]")

**[해석 질문 3] 답:** (seed=26 예시 기준) 팁 평균의 점추정치는 약 2.998달러이고 95% 신뢰구간은 대략 [2.82, 3.18]입니다. 해석하면 "이런 방식으로 구간을 반복해 만들면 그 중 약 95%가 모집단의 참 평균 팁을 포함한다"는 뜻입니다. seed가 다르면 복원추출로 뽑히는 재표본이 달라져 구간이 조금씩 바뀌지만, 부트스트랩은 무작위 재표본에 기반하므로 그 흔들림은 자연스러운 것이며, 구간의 폭과 위치는 seed와 무관하게 서로 비슷하게 나옵니다.

### 1.5 가설검정 — 두 집단 평균차 t검정

In [ ]:
group_dinner = tips.loc[tips["time"] == "Dinner", "tip"]
group_lunch  = tips.loc[tips["time"] == "Lunch",  "tip"]

print("Dinner 평균:", round(group_dinner.mean(), 3), "| n =", len(group_dinner))
print("Lunch  평균:", round(group_lunch.mean(), 3),  "| n =", len(group_lunch))

t_stat, p_val = stats.ttest_ind(group_dinner, group_lunch, equal_var=False)
print(f"\nt 통계량 = {t_stat:.3f}, p-value = {p_val:.4f}")

alpha = 0.05
if p_val < alpha:
    print("결론: 귀무가설 기각 -> 두 시간대의 평균 팁에 유의한 차이가 있다")
else:
    print("결론: 귀무가설 기각 실패 -> 유의한 차이가 있다고 말하기 어렵다")

**해석:** Dinner 평균 팁(≈3.10)이 Lunch(≈2.73)보다 높고, Welch 검정(equal_var=False) p-value는 약 0.041로 유의수준 0.05보다 작습니다. 따라서 위 코드도 "귀무가설 기각 → 두 시간대의 평균 팁에 유의한 차이가 있다"를 출력합니다. 다만 p가 0.05에 꽤 가까운 경계값이므로, 저녁 팁이 더 높다는 증거가 아주 강하지는 않고 표본이 조금만 달라져도 결론이 흔들릴 수 있다는 점에 유의해야 합니다.

In [ ]:
# [해석 질문 4] smoker로 바꿔 확인
g_yes = tips.loc[tips["smoker"] == "Yes", "tip"]
g_no  = tips.loc[tips["smoker"] == "No", "tip"]
t2, p2 = stats.ttest_ind(g_yes, g_no, equal_var=False)
print(f"Smoker Yes 평균={g_yes.mean():.3f}, No 평균={g_no.mean():.3f}")
print(f"t={t2:.3f}, p-value={p2:.4f}")

**[해석 질문 4] 답:** `time` 비교에서는 코드가 "귀무가설 기각(유의한 차이 있음)"을 출력했습니다 — 저녁 팁이 더 높고 p≈0.041로 0.05보다 작기 때문입니다. 다만 0.05에 가까운 경계값이라 여유가 크진 않았습니다. `smoker`로 바꾸면 흡연/비흡연 평균 팁 차이가 매우 작고 p-value가 0.05보다 훨씬 커서(≈0.9) "차이 없음(기각 실패)"으로 나옵니다. 즉 흡연 여부는 팁 평균과 뚜렷한 관련이 없다는 것을 확인할 수 있습니다.

### 1.6 단순선형회귀 (statsmodels OLS)

In [ ]:
model_simple = smf.ols(formula="tip ~ total_bill", data=tips).fit()
print(model_simple.summary())

**[해석 질문 5] 답:** `total_bill`의 계수는 약 0.105입니다. 즉 "청구액이 1달러 오르면 팁이 평균 약 0.105달러(약 10센트) 오른다"고 해석합니다. 이 계수의 p-value는 사실상 0(<0.001)으로 매우 유의하며, 95% 신뢰구간은 대략 [0.09, 0.12]로 0을 포함하지 않습니다. 절편은 약 0.92로, 청구액이 0일 때의 팁(외삽값이라 실질 의미는 제한적)입니다. R²는 약 0.457로 청구액만으로 팁 변동의 약 46%를 설명합니다.

### 1.7 잔차 진단

In [ ]:
fitted = model_simple.fittedvalues
resid  = model_simple.resid

fig, axes = plt.subplots(1, 2, figsize=(11,4))

axes[0].scatter(fitted, resid, alpha=0.5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("예측값(fitted)")
axes[0].set_ylabel("잔차(resid)")
axes[0].set_title("잔차 플롯")

sm.qqplot(resid, line="45", fit=True, ax=axes[1])
axes[1].set_title("정규 Q-Q 플롯")
plt.show()

**[해석 질문 6] 답:** 잔차 플롯에서 예측값이 커질수록 잔차의 위아래 퍼짐이 넓어지는 깔때기(fan) 모양이 보입니다 — **등분산 가정 위반(이분산)**의 신호입니다. Q-Q 플롯도 오른쪽 꼬리에서 점들이 45도 직선 위로 벗어나, 잔차가 완전한 정규분포는 아니고 오른쪽으로 치우쳐 있음을 보여 줍니다. 이는 큰 청구액에서 팁 변동이 커지기 때문이며, 로그변환(심화 2.3)이나 로버스트 표준오차로 완화할 수 있습니다.

### 1.8 다중회귀 (범주형 one-hot 포함)

In [ ]:
formula_multi = "tip ~ total_bill + size + C(sex) + C(smoker) + C(day) + C(time)"
model_multi = smf.ols(formula=formula_multi, data=tips).fit()
print(model_multi.summary())

**해석:** `C(...)`가 범주형을 자동으로 원-핫 처리했습니다. statsmodels는 각 범주형에서 첫 범주를 기준(baseline)으로 빼고, 나머지를 `T.<범주>` 형태의 계수로 표시합니다 — 그 계수는 '기준 대비 차이'를 뜻합니다. 여기서 빠진 기준범주는 **sex의 Male(계수는 T.Female), time의 Lunch(계수는 T.Dinner), smoker의 Yes(계수는 T.No), day의 Thur** 등입니다. 대부분의 범주형 계수는 p-value가 커서 유의하지 않고, 수치형 `total_bill`이 압도적으로 유의합니다(`size`는 뒤에서 보듯 0.05 경계).

### 1.9 계수 유의성 · R² / Adj R² · VIF

In [ ]:
print(f"단순회귀 R^2 = {model_simple.rsquared:.4f}, Adj R^2 = {model_simple.rsquared_adj:.4f}")
print(f"다중회귀 R^2 = {model_multi.rsquared:.4f}, Adj R^2 = {model_multi.rsquared_adj:.4f}")

In [ ]:
vif_df = tips[["total_bill", "size"]].copy()
vif_df = sm.add_constant(vif_df)

vif_result = pd.DataFrame()
vif_result["변수"] = vif_df.columns
vif_result["VIF"] = [variance_inflation_factor(vif_df.values, i) for i in range(vif_df.shape[1])]
# 주의: const(상수항) 행 VIF는 절편 때문에 생기는 산물이라 무시하고 total_bill·size만 해석
print(vif_result)

**[해석 질문 7] 답:** 질문은 "p-value가 0.05보다 작아 유의한 변수"를 묻습니다. 이 기준을 엄격히 적용하면 다중회귀에서 유의한 변수는 **`total_bill` 하나뿐입니다(p<0.001)**. `size`는 p≈0.051로 0.05를 아슬하게 넘겨(거의 유의한 경계값) 0.05 기준에서는 유의하다고 보지 않습니다. `sex`/`smoker`/`day`/`time`의 더미들은 p-value가 커서 유의하지 않습니다. `total_bill`과 `size`의 VIF는 각각 약 1.6 내외로 5보다 훨씬 작아 다중공선성 걱정은 없습니다(상수항의 VIF 값은 무시). Adj R²는 단순회귀(≈0.4544) → `total_bill+size`(≈0.4635)까지는 소폭 올랐지만, 범주형까지 모두 넣은 full 모델에서는 오히려 **약 0.4520으로 소폭 하락**합니다. 이는 추가한 범주형 변수들이 팁 설명에 실질적 기여를 못 해, Adj R²의 변수 개수 벌점만 받았다는 뜻입니다(불필요한 변수는 Adj R²를 깎는다).

### 1.10 해석 vs 예측 (train/test RMSE · MAE)

In [ ]:
# 목표변수 tip이 X에 새어들지 않도록 제거.
# (심화 2.3을 먼저 돌렸다면 log_tip 열이 생겨 있을 수 있어 함께 방어적으로 제거 — 정상 순서라면 아직 없어 무시됨)
features = tips.drop(columns=["tip", "log_tip"], errors="ignore")
X = pd.get_dummies(features, drop_first=True).astype(float)
y = tips["tip"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=MY_SEED
)
print("train:", X_train.shape, "| test:", X_test.shape)

In [ ]:
lin = LinearRegression()
lin.fit(X_train, y_train)

pred_test = lin.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred_test))
mae  = mean_absolute_error(y_test, pred_test)
print(f"Test RMSE = {rmse:.3f}")
print(f"Test MAE  = {mae:.3f}")

**[해석 질문 8] 답:** (seed=26 예시) Test RMSE는 약 1.15, MAE는 약 0.81 수준으로 **RMSE가 MAE보다 큽니다**. RMSE는 오차를 제곱해 평균하므로 큰 오차(예: 청구액이 큰데 팁이 유난히 튀는 손님)에 더 민감하고, 그래서 그런 큰 오차가 있으면 MAE보다 값이 커집니다. 두 지표의 목적 차이를 정리하면: statsmodels의 R²/계수/p-value는 "어떤 변수가 팁과 어떻게 연관되는가"를 **설명·해석**하는 데 초점이 있고, train/test로 나눠 잰 RMSE/MAE는 "이 모델이 본 적 없는 새 손님의 팁을 얼마나 정확히 맞히는가"를 **예측 성능**으로 평가합니다. 같은 선형회귀라도 던지는 질문이 다릅니다. (구체적 수치는 `MY_SEED`에 따라 조금씩 달라집니다.)

### 1.11 결론

1. **추론:** 팁 평균의 95% 신뢰구간은 대략 [2.8, 3.2]달러였고, 시간대(점심/저녁)별 팁 차이는 p≈0.041로 0.05보다 작아 (경계이긴 하지만) 유의하게 기각되었으며, 흡연 여부에 따른 차이는 유의하지 않았습니다. 즉 저녁 팁이 다소 높다는 신호는 있으나 그 증거는 강하지 않습니다.
2. **회귀:** 단순회귀에서 청구액 1달러당 팁이 약 0.105달러 증가(매우 유의, R²≈0.46). 다중회귀에서도 p<0.05로 엄밀히 유의한 변수는 `total_bill` 하나뿐이며(`size`는 p≈0.051 경계), 범주형 변수를 모두 넣은 full 모델은 Adj R²가 오히려 소폭 하락(0.464→0.452)했습니다. VIF상 공선성 문제도 없었습니다.
3. **해석 vs 예측:** statsmodels(해석)는 "청구액이 핵심 동인"임을 알려 주고, sklearn train/test(예측)는 그 모델이 새 데이터에서 약 RMSE≈1.15로 팁을 맞힌다는 실사용 성능을 알려 줍니다. 두 관점은 상충하지 않고 상호보완적입니다.

## 2. 심화 과제

### 2.1 여러 seed로 신뢰구간 반복

In [ ]:
seeds = [MY_SEED + k for k in range(10)]
intervals = []
for s in seeds:
    r = np.random.default_rng(s)
    bmeans = np.array([r.choice(tip_values, size=len(tip_values), replace=True).mean() for _ in range(5000)])
    lo, hi = np.percentile(bmeans, [2.5, 97.5])
    intervals.append((s, lo, hi))

for s, lo, hi in intervals:
    print(f"seed={s}: [{lo:.3f}, {hi:.3f}]")

**[심화 질문] 답:** 10개 seed의 구간은 소수점 둘째 자리에서 조금씩 다르지만 폭(≈0.35)과 위치(≈2.82~3.18)가 거의 같고 서로 대부분 겹칩니다. 부트스트랩 신뢰구간이 무작위 재표본에 기반하더라도 결론이 seed에 좌우되지 않을 만큼 안정적이라는 뜻이며, 그래서 신뢰구간을 실제 추론 근거로 사용할 수 있습니다.

### 2.2 변수 추가/제거로 R²·VIF 변화

In [ ]:
formulas = {
    "total_bill만": "tip ~ total_bill",
    "total_bill+size": "tip ~ total_bill + size",
    "full": formula_multi,
}
for name, f in formulas.items():
    m = smf.ols(f, data=tips).fit()
    print(f"{name:20s} | Adj R^2 = {m.rsquared_adj:.4f}")

**해석:** `size`를 추가하면 Adj R²가 소폭 상승(≈0.4544→0.4635)하지만, 범주형까지 모두 넣은 full 모델은 Adj R²가 오히려 **소폭 하락(≈0.4520)**합니다. 즉 유용한 변수는 청구액과 일행 수 정도이고, 나머지 범주형은 Adj R²의 벌점(변수 개수 페널티)만 늘려 값을 떨어뜨립니다 — "불필요한 변수를 넣으면 Adj R²가 내려간다"는 교과서적 예시입니다.

### 2.3 로버스트 / 로그변환

In [ ]:
tips["log_tip"] = np.log(tips["tip"])
model_log = smf.ols("log_tip ~ total_bill + size", data=tips).fit()

plt.figure(figsize=(5,4))
plt.scatter(model_log.fittedvalues, model_log.resid, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.title("로그변환 후 잔차 플롯")
plt.show()

print("로그변환 모델 R^2(Adj):", round(model_log.rsquared_adj, 4))

**해석:** 종속변수를 `log(tip)`으로 바꾸면 잔차 플롯의 깔때기 모양(이분산)이 눈에 띄게 완화되어 위아래 퍼짐이 더 고르게 됩니다. 다만 로그변환 모델의 계수는 "청구액 1달러당 팁이 몇 % 변한다" 식(로그-선형)으로 해석이 달라지므로, 해석 목적에 맞게 변환 여부를 선택해야 합니다. (주의: `log_tip` 열은 여기서 생성되므로, 이 셀을 돌린 뒤 1.10을 재실행할 때 목표 누수를 막기 위해 1.10의 `features`에서 `log_tip`을 제거해 둔 것입니다.)

## 3. 회고

**예시:** 가장 어려웠던 부분은 "해석용 모델(statsmodels)"과 "예측용 모델(sklearn)"이 왜 따로 필요한가였습니다. 처음엔 R²만 보면 되는 줄 알았는데, R²는 가진 데이터를 얼마나 설명하는지일 뿐 새 데이터 예측력과는 다르다는 점을 train/test RMSE를 직접 계산하며 이해했습니다. 다중회귀에서 `size`의 p-value가 0.051로 0.05를 아슬하게 넘겨 "경계값"을 어떻게 서술해야 하는지도 고민이었고, 유의성은 0.05를 딱 자르기보다 여유 정도를 함께 봐야 한다는 걸 배웠습니다. 아직 t검정에서 등분산 가정(equal_var 옵션)의 선택 기준은 더 공부가 필요합니다.

## 4. 생성형 AI 활용 방법

**활용 방법 요약 (예시):** statsmodels formula에서 범주형을 `C()`로 넣는 문법과 VIF 계산 코드 골격을 AI에게 물어보고, 반환된 코드가 실제 `tips` 데이터에서 오류 없이 돌아가는지, 계수 부호/크기가 EDA에서 본 상관 방향과 일치하는지를 직접 대조해 검증했습니다. 또 AI는 `size`를 "유의한 변수"라고 단정했지만, `.summary()`의 p-value를 직접 확인해 보니 0.051로 0.05 경계여서 "엄밀히는 유의하지 않다"로 정정했습니다. AI가 `mean_squared_error(..., squared=False)`를 제안했으나 sklearn 버전에 따라 동작이 다를 수 있어, 대신 `np.sqrt(mean_squared_error(...))`로 바꿔 안전하게 처리했습니다.

```
(실제 제출본에는 학습자가 사용한 프롬프트와 답변을 붙여넣습니다.)
```